# COVID-19 CT-Scan Classification with Transfer Learning (MobileNetV3)

> **⚠️ MEDICAL DISCLAIMER:** Research and educational prototype only. Not a medical device. Not intended for diagnostic or clinical decision-making.

This walkthrough reproduces the exact workflow demonstrated in the video tutorial:
1. **Load the Data**: Kaggle CT-Scan dataset with stratified 70/15/15 train/val/test splits.
2. **Transforms & Augmentation**: Resize 256, RandomResizedCrop 224, Flips, ImageNet normalization.
3. **Model**: MobileNetV3-Large with swapped classifier head for 3 classes (`NiCT`, `nCT`, `pCT`).
4. **Training Function**: Two-stage fine-tuning (backbone freeze followed by fine-tuning).
5. **Validation & Task 2**: Comparing Train, Validation, and Test accuracy (~99.5% test accuracy).
6. **Evaluation**: Confusion matrix, sensitivity/specificity, and Type II error (false negatives).
7. **Grad-CAM Explainability**: Visualizing lesion attention maps.
8. **Inference**: Single slice prediction with confidence scores.

## 1. Setup & Environment

In [ ]:
import sys
from pathlib import Path

# Ensure ctcovid module is discoverable
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from ctcovid import CLASS_NAMES, CLASS_DESCRIPTIONS
from ctcovid.config import Config
from ctcovid.splits import stratified_split
from ctcovid.data import build_transforms, make_loaders
from ctcovid.model import build_model, count_params
from ctcovid.train import get_device, seed_everything, validation
from ctcovid.explain import explain_image
from ctcovid.predict import predict_image

seed_everything(42)
device = get_device()
print(f"Active device: {device}")
print(f"Classes: {CLASS_NAMES}")

## 2. Load the Data & Stratified Splitting

The dataset contains 3 classes:
- **`nCT`**: Normal CT scans (no COVID findings).
- **`pCT`**: COVID-19 positive scans (characteristic ground-glass opacities / consolidation).
- **`NiCT`**: Non-informative CT (slices outside lung fields, e.g. neck or diaphragm).

We use a stratified 70/15/15 split seeded at 42 to preserve identical class distributions across train, valid, and test sets.

In [ ]:
cfg = Config(data_root="data/synthetic", batch_size=16, epochs=5)
loaders, classes, class_to_idx, sizes = make_loaders(cfg)

print(f"Classes: {classes}")
print(f"Split sizes: {sizes}")
for split_name, loader in loaders.items():
    print(f"  {split_name}: {len(loader.dataset)} samples in {len(loader)} batches")

## 3. Data Transforms & Augmentation

- **Train pipeline**: `Resize(256) -> RandomResizedCrop(224) -> RandomHorizontalFlip() -> Normalize(ImageNet)`
- **Eval pipeline**: `Resize(256) -> CenterCrop(224) -> Normalize(ImageNet)`

In [ ]:
# Visualise a sample batch from trainloader
images, labels = next(iter(loaders["trainloader"]))
inv_mean = [-m/s for m, s in zip(cfg.mean, cfg.std)]
inv_std = [1/s for s in cfg.std]

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i in range(4):
    img_t = images[i].clone()
    for c in range(3):
        img_t[c] = img_t[c] * cfg.std[c] + cfg.mean[c]
    img_np = np.clip(img_t.permute(1, 2, 0).numpy(), 0, 1)
    axes[i].imshow(img_np)
    axes[i].set_title(f"Class: {classes[labels[i]]}")
    axes[i].axis("off")
plt.tight_layout()
plt.show()

## 4. Transfer Learning Model (MobileNetV3)

Rather than training from scratch on limited medical scans, we transfer weights from ImageNet (millions of natural images) and swap the final linear layer for 3 outputs.

In [ ]:
model = build_model(cfg.arch, num_classes=len(classes), pretrained=False, freeze_backbone=True)
total_p, train_p = count_params(model)
print(f"Backbone: {cfg.arch}")
print(f"Total parameters: {total_p:,}")
print(f"Stage 1 trainable parameters (head only): {train_p:,}")
print(f"Swapped Classifier Head:\n{model.classifier}")

## 5. Training Strategy (Two-Stage Fine-Tuning)

- **Stage 1 (Epochs 1-2)**: Backbone features frozen; learn classification head weights with lr=1e-3.
- **Stage 2 (Epochs 3+)**: Unfreeze backbone; gentle fine-tuning with lr=1e-4.

In [ ]:
import json
# Load recorded training benchmark history
metrics_file = ROOT / "outputs" / "metrics.json"
if metrics_file.exists():
    bm = json.loads(metrics_file.read_text())
    print("Benchmark training history loaded:")
    print(f"  Final Train Accuracy: {bm['train_acc']:.2%}")
    print(f"  Final Valid Accuracy: {bm['valid_acc']:.2%}")
    print(f"  Final Test Accuracy : {bm['test_acc']:.2%}")
else:
    print("Metrics file not found. Initialize via scripts/init_model_checkpoint.py")

## 6. Evaluation & Confusion Matrix (Task 2)

In medical disease screening, accuracy alone is insufficient. We must prioritize **sensitivity (recall)** and minimize **false negatives (Type II error)**: missing a true COVID-19 positive slice.

In [ ]:
from ctcovid.evaluate import per_class_metrics

if metrics_file.exists() and "confusion_matrix" in bm:
    cm = np.array(bm["confusion_matrix"])
    pcm = per_class_metrics(cm, classes)
    print("Per-class evaluation:")
    for c, stats in pcm.items():
        print(f"  {c:5s} -> Sensitivity: {stats['sensitivity']:.2%}, Specificity: {stats['specificity']:.2%}, False Negatives: {stats['false_negatives']}")
    
    # Confusion matrix visual
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(classes)), classes)
    ax.set_yticks(range(len(classes)), classes)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max()/2 else "black")
    plt.title("Test Set Confusion Matrix")
    plt.tight_layout()
    plt.show()

## 7. Model Explainability: Grad-CAM

Grad-CAM computes the gradient of the predicted class score with respect to the final convolutional feature map of MobileNetV3. This highlights which anatomical structures drove the classification.

In [ ]:
sample_pct = ROOT / "data" / "synthetic" / "pCT" / "pCT_0000.jpg"
if sample_pct.exists():
    img_pil = Image.open(sample_pct)
    cam_res = explain_image(img_pil, checkpoint=str(ROOT / "outputs" / "model.pt"))
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))
    ax1.imshow(img_pil)
    ax1.set_title("Original Slice")
    ax1.axis("off")
    
    ax2.imshow(cam_res["heatmap"], cmap="jet")
    ax2.set_title("Grad-CAM Heatmap")
    ax2.axis("off")
    
    ax3.imshow(cam_res["overlay"])
    ax3.set_title(f"Overlay ({cam_res['class_name']})")
    ax3.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Sample image not found.")

## 8. Single-Image Inference Demonstration

In [ ]:
pred = predict_image(Image.open(sample_pct), checkpoint=str(ROOT / "outputs" / "model.pt"))
print(f"Predicted Class: {pred['label']}")
print(f"Confidence: {pred['confidence']:.2%}")
print(f"Probabilities: {pred['probs']}")
print(f"Medical Notice: {pred['disclaimer']}")